# Interactive Portrait Shadow Removal

**Architecture:** ResNet-34 U-Net + VGG16 Perceptual Loss
This notebook sets up the environment, prepares the dataset, and handles model training and evaluation.

### Step 1: Mount Google Drive
Mount drive to save datasets and model checkpoints.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Ensure this path matches the exact folder location in your individual Google Drive
project_path = '/content/drive/MyDrive/Interactive_Shadow_Remover'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Current working directory successfully changed to: {os.getcwd()}")
else:
    print(f"CRITICAL: Folder {project_path} not found. Please verify your Google Drive structure.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory successfully changed to: /content/drive/MyDrive/Interactive_Shadow_Remover


### Step 2: Install Dependencies
Install required libraries for training and UI.

In [2]:
!pip install -r requirements.txt
!pip install ninja scikit-image streamlit==1.28.0 lpips streamlit-drawable-canvas diffusers transformers accelerate datasets facenet-pytorch torchvision

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached certifi-2024.12.14-py3-none-any.whl.metadata (2.3 kB)
  Using cached charset_normalizer-3.4.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (34 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached filelock-3.16.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)
  Using cached fsspec-2024.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached grpcio-1.68.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.9 kB)
  Using cached h5py-3.12.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.5 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached jinja2-3.1.5-py3-none-any.whl.metadata (2.6 kB)
  Using cached keras-3.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached Mark

### Step 3: Download Dataset
Download high-resolution face images (CelebA-HQ and FFHQ) from HuggingFace.

In [3]:
!python src/download_data.py

Initializing Robust Dataset Collection (Target: 500 Faces)...

Fetching 250 images from 'bitmind/ffhq-256'...
README.md: 100% 288/288 [00:00<00:00, 1.39MB/s]
  -> Downloaded 50/250 images...
  -> Downloaded 100/250 images...
  -> Downloaded 150/250 images...
  -> Downloaded 200/250 images...
  -> Downloaded 250/250 images...
✅ Successfully completed! Saved 250 samples from bitmind/ffhq-256.

Fetching 250 images from 'korexyz/celeba-hq-256x256'...
README.md: 100% 989/989 [00:00<00:00, 4.29MB/s]
  -> Downloaded 50/250 images...
  -> Downloaded 100/250 images...
  -> Downloaded 150/250 images...
  -> Downloaded 200/250 images...
  -> Downloaded 250/250 images...
✅ Successfully completed! Saved 250 samples from korexyz/celeba-hq-256x256.

DATA COLLECTION COMPLETE! Total combined portraits ready for synthesis: 500
Fatal Python error: PyGILState_Release: thread state 0x7f133be46bb0 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007f144

### Step 4: Generate Synthetic Shadows
Create synthetic shadow triplets (Shadow Image, Mask Image, Clean Ground Truth) for training.

In [4]:
!python src/synthetic_data_gen.py

Dataset generated! 500 perfectly structured image triplet pairs ready for Pix2Pix U-Net mapping.


### Step 5: Train U-Net Model
Train the ResNet34 U-Net model using VGG perceptual loss.
Note: Training takes ~30-40 minutes on a T4 GPU.

In [5]:
!python src/train_unet.py --epochs 50 --batch_size 16

Training on Device: cuda
100% 107M/107M [00:00<00:00, 123MB/s]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:04<00:00, 115MB/s] 
Starting Training for 50 Epochs...
Epoch [1/50] -> Average Compound Loss: 1.1807
>>> Saved New Best Model!
Epoch [2/50] -> Average Compound Loss: 0.8369
>>> Saved New Best Model!
Epoch [3/50] -> Average Compound Loss: 0.6639
>>> Saved New Best Model!
Epoch [4/50] -> Average Compound Loss: 0.5452
>>> Saved New Best Model!
Epoch [5/50] -> Average Compound Loss: 0.4709
>>> Saved New Best Model!
Epoch [6/50] -> Average Compound Loss: 0.4204
>>> Saved New Best Model!
Epoch [7/50] -> Average Compound Loss: 0.3959
>>> Saved New Best Model!
Epoch [8/50] -> Average Compound Loss: 0.3741
>>> Saved New Best Model!
Epoch [9/50] -> Average Compound Loss: 0.3629
>>> Saved New Best Model!
Epoch [10/50] -> Average Compound Loss: 0.3558
>>> Saved New Best Model!
Epoch [11/50] -

### Step 6: Evaluation Metrics
Calculate PSNR, SSIM, and identity loss on the validation set.

In [6]:
!python evaluate/ablation_study.py

Running evaluation on cuda
Evaluating 50 samples...
100% 50/50 [00:07<00:00,  7.04it/s]


       Quantitative Analysis: Shadow Removal Performance        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ Configuration            ┃ PSNR (dB) ↑ ┃ SSIM ↑ ┃ ID Error ↓ ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ Baseline (StyleGAN)      │    17.61    │ 0.8066 │  0.00066   │
│ Ours (U-Net Restoration) │    23.63    │ 0.8418 │  0.00018   │
└──────────────────────────┴─────────────┴────────┴────────────┘

Visuals saved to /visual_comparisons/


### Step 7: Run Streamlit UI
Start the interactive web interface.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

!streamlit run ui/app.py --server.enableCORS false --server.enableXsrfProtection false